# Kimi K3 · M9.4
### Qualitative evaluation laboratory

**294.7M parameters · 6.0B training tokens · FP32 · KDA chunkwise · RTX 4050**

Use this notebook to interactively explore **base-language-model continuations** and run the separate, **pre-registered twelve-case qualitative battery**. Exploratory outputs are not evidence of benchmark performance. Scientific results should include unedited good and bad outputs.

**Known limitations:** M9.2 verified FP32 causal suffix-invariance but not BF16; the KDA chunkwise evaluation backend has not been numerically matched to the fused KDA used during pretraining. This notebook is deliberately no-cache, so generation gets slower as prefixes grow. Prompts + generated tokens must fit within **1,024 tokens**; generation is capped at **64 new tokens**.

## 1 · Environment and frozen contract

Execute these two cells once. Preflight is read-only and does **not** load the checkpoint. All paths resolve from the repository, wherever Jupyter starts.

In [1]:
from pathlib import Path
import sys, json
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
parents = [cwd, *cwd.parents]
repo = next((p for p in parents if (p/'scripts/m9/m94_core.py').is_file()), None)
if repo is None:
    raise RuntimeError('Notebook must run within the kimi-k3-from-scratch checkout (or launch Jupyter from repo root).')
if str(repo) not in sys.path:
    sys.path.insert(0,str(repo))
from scripts.m9 import m94_core as m94
print('Repository:', repo)
print('Checkpoint:', m94.CHECKPOINT)
print('Output folder:', m94.OUT)


Repository: C:\Users\aflon\OneDrive\Documentos\GitHub\kimi-k3-from-scratch
Checkpoint: C:\datasets\kimi_k3_backups\m8\m8_full_6b_v1\snapshot_step_183106.pt
Output folder: C:\Users\aflon\OneDrive\Documentos\GitHub\kimi-k3-from-scratch\results\m9\m94_qualitative


In [2]:
contract, fixed_protocol = m94.verify_preflight()
print('PREFLIGHT: PASS')
print('Fixed cases:', len(fixed_protocol['cases']))
print('Protocol SHA-256:', m94.FROZEN_M94_PROTOCOL_SHA256)
print('Checkpoint SHA-256: inherited from independently verified M9.1; file size checked here')
print('Model backend:', fixed_protocol['backend'], '| precision:', fixed_protocol['precision'])


PREFLIGHT: PASS
Fixed cases: 12
Protocol SHA-256: 7f2dedd49dc09689789b95570f66afd02382b07fcf11852c3de0b7bde781f9c8
Checkpoint SHA-256: inherited from independently verified M9.1; file size checked here
Model backend: chunkwise | precision: fp32


## 2 · Load Kimi K3 once

The 2.49-GB trainer checkpoint is read through restricted `torch.load` with `mmap=True`. Only `model` tensors are copied into a strictly matched model. The optimizer, training cursors, RNG and trainer are **not restored**. Use `device='cuda'` on your RTX 4050; set `local_files_only=True` if the tokenizer is already cached and you want to prevent downloads.

In [3]:
import time, torch

MODEL_DEVICE = 'cuda'    # or 'cpu' (slow)
LOCAL_TOKENIZER_ONLY = False

t0 = time.perf_counter()
session = m94.load_session(device=MODEL_DEVICE, local_files_only=LOCAL_TOKENIZER_ONLY)
print(f"Model ready in {time.perf_counter()-t0:.2f}s")
print('Device:', session['device'])
print('Parameters:', sum(p.numel() for p in session['model'].parameters()))
if session['device'].type=='cuda':
    print('GPU:', torch.cuda.get_device_name(session['device']))
    print(f"Allocated after load: {torch.cuda.memory_allocated()/2**20:.1f} MiB")


Model ready in 13.88s
Device: cuda
Parameters: 294676808
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Allocated after load: 1127.7 MiB


## 3 · Playground

Edit the parameters below and rerun the cell as many times as you want. This is **exploration**: changing the prompt or sampling configuration does not change the frozen scientific battery. For `sample`, temperature, top-k and top-p change how the next token is drawn. `greedy` always takes the most likely token.

In [6]:
PROMPT = 'We are made of'
MODE = 'greedy'              # 'greedy' or 'sample'
MAX_NEW_TOKENS = 32         # 1..64
TEMPERATURE = 1.0
TOP_K = 50
TOP_P = 0.95
SEED = 1337

play = m94.generate_case(session, prompt=PROMPT, mode=MODE, max_new_tokens=MAX_NEW_TOKENS,
                          temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P, seed=SEED)
print('PROMPT:', repr(play['prompt']))
print('CONTINUATION:', repr(play['continuation_text']))
print('FULL TEXT:', play['full_text'])
print(f"Tokens: {play['generated_token_count']} | EOS: {play['eos_hit']} | "
      f"Repeated trigrams: {play['repeated_trigram_fraction']:.3f} | Seconds: {play['seconds_generation']:.2f}")
print('First-step top-5 predictions:')
for item in play['first_step_top5']:
    print(f"  {item['token_id']:5}  {item['token']!r:20}  p={item['prob']:.4f}")


PROMPT: 'We are made of'
CONTINUATION: ' 100% pure water.\n- Water is the only substance that can be used for cooking.\n- Water is the only substance that can be'
FULL TEXT: We are made of 100% pure water.
- Water is the only substance that can be used for cooking.
- Water is the only substance that can be
Tokens: 32 | EOS: False | Repeated trigrams: 0.300 | Seconds: 5.37
First-step top-5 predictions:
    216  ' '                   p=0.0915
    800  ' many'               p=0.0743
    260  ' the'                p=0.0610
    253  ' a'                  p=0.0418
    913  ' water'              p=0.0304


### Optional GUI controls

The following cell creates a text box and controls with a **Generate** button if `ipywidgets` is installed. It never runs a generation automatically. Clicking **Save exploratory result** creates a separately labeled file; it does **not** add evidence to the fixed evaluation. If widgets are missing, the editable playground cell above remains fully functional.

In [7]:
try:
    import ipywidgets as W
    from IPython.display import clear_output
    from uuid import uuid4
except ImportError:
    print('ipywidgets is not installed. Use the editable playground cell above, or launch JupyterLab with --with ipywidgets.')
else:
    w_prompt = W.Textarea(value=PROMPT, description='Prompt', layout=W.Layout(width='100%', height='85px'), style={'description_width':'70px'})
    w_mode = W.Dropdown(options=['greedy','sample'], value='greedy', description='Mode')
    w_new = W.IntSlider(value=32,min=1,max=64,description='New tokens',continuous_update=False)
    w_temp = W.FloatSlider(value=1.0,min=0.2,max=2.0,step=0.1,description='Temp',continuous_update=False)
    w_k = W.IntSlider(value=50,min=1,max=200,description='Top-k',continuous_update=False)
    w_p = W.FloatSlider(value=.95,min=.1,max=1.0,step=.05,description='Top-p',continuous_update=False)
    w_seed = W.IntText(value=1337,description='Seed')
    btn_run = W.Button(description='Generate continuation',button_style='primary',icon='play')
    btn_save = W.Button(description='Save exploratory result',button_style='')
    ui_out = W.Output()
    ui_state={'last':None}
    def on_run(_):
        with ui_out:
            clear_output(wait=True)
            try:
                r=m94.generate_case(session,prompt=w_prompt.value,mode=w_mode.value,max_new_tokens=w_new.value,
                        temperature=w_temp.value,top_k=w_k.value,top_p=w_p.value,seed=w_seed.value)
                ui_state['last']=r
                print('Prompt:',repr(r['prompt']))
                print('Continuation:',repr(r['continuation_text']))
                print('Full text:\n'+r['full_text'])
                print(f"\nTokens {r['generated_token_count']} | EOS {r['eos_hit']} | "
                      f"Repeated trigrams {r['repeated_trigram_fraction']:.3f} | {r['seconds_generation']:.2f}s")
            except Exception as exc:
                ui_state['last']=None
                print(f'ERROR: {type(exc).__name__}: {exc}')
    def on_save(_):
        with ui_out:
            r=ui_state['last']
            if r is None:
                print('Nothing to save: generate something first.')
                return
            file = m94.OUT/'playground'/('exploratory_'+uuid4().hex[:12]+'.json')
            m94.atomic_json(file,r)
            print('Exploratory output saved:',file)
    btn_run.on_click(on_run)
    btn_save.on_click(on_save)
    display(W.VBox([w_prompt,W.HBox([w_mode,w_new]),W.HBox([w_temp,w_k,w_p]),w_seed,
                     W.HBox([btn_run,btn_save]),ui_out]))


## 4 · How do decoding parameters change the text?

Optional experiment, deliberately outside the frozen battery. The code will generate the **same starting prompt** using several settings. This section is not pre-registered evidence and may be run repeatedly. Flip `RUN_COMPARISON` to `True` to execute.

In [9]:
RUN_COMPARISON = True
COMPARE_PROMPT = 'The results of the experiment showed that'
if RUN_COMPARISON:
    strategies=[('greedy','greedy',1.0),('sample_0.7','sample',0.7),
                ('sample_1.0','sample',1.0),('sample_1.3','sample',1.3)]
    comparisons=[]
    for label,mode,temp in strategies:
        result=m94.generate_case(session,prompt=COMPARE_PROMPT,mode=mode,
                                 temperature=temp,max_new_tokens=32,seed=1337,top_k=50,top_p=.95)
        comparisons.append((label,result))
        print('\n',label.upper(),':',repr(result['continuation_text']))
else:
    print('Set RUN_COMPARISON=True to compare strategies.')



 GREEDY : ' the number of people who were able to identify the location of a person’s body was significantly higher than the number of people who were able to identify the location'

 SAMPLE_0.7 : ' the number of birds that were fed a diet of seeds was significantly greater than the number that were fed the seeds.\nThe authors concluded that seeds could be used'

 SAMPLE_1.0 : ' the number of birds released was only 20,000 in comparison to birds released 25000 miles apart. The numbers of birds'

 SAMPLE_1.3 : ' the temperature in the two compartments on the sides of the tank was reduced as 2 cm greater width of tube was added. However, the temperature outside the tank'


## 5 · Fixed pre-registered battery (scientific evidence)

Here are **all** the six frozen English prefixes, each evaluated once using greedy and once with seeded sampling (12 cases, max 48 tokens each). The protocol JSON was hashed before generated results existed. Do not edit that file. You must explicitly set `RUN_FIXED = True`; **Run All** won't start the battery by default. The runner saves each completed case atomically. To resume after an interruption, set `RESUME_FIXED = True` (do not clear the results).

In [10]:
from collections import defaultdict
fixed_pairs=defaultdict(list)
for case in fixed_protocol['cases']:
    fixed_pairs[case['group']].append(case)
for group, cases in fixed_pairs.items():
    print(f"{group}: {cases[0]['prompt']!r}  →  {', '.join(c['mode'] for c in cases)}")
print('Pre-registered SHA-256:',m94.FROZEN_M94_PROTOCOL_SHA256)


narrative: 'Once upon a time, in a small village'  →  greedy, sample
scientific prose: 'The results of the experiment showed that'  →  greedy, sample
expository prose: 'The main advantage of solar energy is'  →  greedy, sample
historical prose: 'During the nineteenth century, many cities'  →  greedy, sample
fiction continuation: 'When Alice opened the door, she saw'  →  greedy, sample
procedural prose: 'To prepare the mixture, first add'  →  greedy, sample
Pre-registered SHA-256: 7f2dedd49dc09689789b95570f66afd02382b07fcf11852c3de0b7bde781f9c8


In [ ]:
RUN_FIXED = False      # Change to True only when ready
RESUME_FIXED = False   # Change to True if a previous fixed run exists, even if interrupted

if RUN_FIXED:
    fixed_summary = m94.run_fixed(session, out=m94.OUT/'fixed', resume=RESUME_FIXED)
    print('STATUS:',fixed_summary['status'])
    print('Results:',m94.OUT/'fixed/m94_results.json')
    print('Report:',m94.OUT/'fixed/M94_READOUT.md')
else:
    print('Frozen battery not started. Set RUN_FIXED=True when ready.')


## 6 · Review every fixed continuation (not just the best ones)

This cell reads existing fixed results if present. It does not generate anything. Review each pair and consider: relevance to prefix, grammar, within-paragraph coherence, repetition/degeneration, abrupt topic changes, obvious factual errors, and early EOS. No metric here automatically measures "quality".

In [ ]:
fixed_file=m94.OUT/'fixed/m94_results.json'
if fixed_file.is_file():
    result=json.loads(fixed_file.read_text(encoding='utf-8'))
    print('FIXED STATUS:',result['status'],'| cases:',len(result['cases']))
    for r in result['cases']:
        print('\n'+'='*76)
        print(f"{r['case_id']}   {r['group']}   {r['mode']}   seed {r['seed']}")
        print('PROMPT:',repr(r['prompt']))
        print('CONTINUATION:',repr(r['continuation_text']))
        print(f"EOS: {r['eos_hit']} | Tokens: {r['generated_token_count']} | "
              f"Repeat-3: {r['repeated_trigram_fraction']:.3f}")
else:
    print('No frozen results yet. Run the battery in section 5 or via scripts/m9/m94_run.py --fixed.')


## 7 · Descriptive diagnostics (not quality scores)

The plots show mechanical repetition and generated-token diversity, separately for greedy and sampling. They cannot certify factuality, intelligence, or coherence.

In [ ]:
if fixed_file.is_file():
    import matplotlib.pyplot as plt
    r=json.loads(fixed_file.read_text(encoding='utf-8'))
    names=[x['case_id'] for x in r['cases']]
    rep=[x['repeated_trigram_fraction'] for x in r['cases']]
    fig,ax=plt.subplots(figsize=(12,4))
    ax.bar(names,rep)
    ax.set_title('Repeated trigram fraction (per continuation)')
    ax.set_ylabel('Fraction of repeated 3-token n-grams')
    ax.set_ylim(0,1)
    ax.tick_params(axis='x',labelrotation=70)
    plt.tight_layout()
    plt.show()
else:
    print('No fixed run recorded yet.')


## 8 · Human notes for M9.5

After inspecting all results, add qualitative observations here, including failures. You can save a separate human-review file; it won't alter the frozen prompts or raw generations.

In [ ]:
HUMAN_NOTES = {
    'strong_examples': [],        # e.g. case IDs and what worked well
    'failure_examples': [],       # e.g. case IDs and failure mechanism
    'comparison_greedy_vs_sample': '',
    'generalization_limitations': 'These are base-LM continuations, not a chat/instruct evaluation.',
    'notes': '',
}
SAVE_HUMAN_NOTES = False
if SAVE_HUMAN_NOTES:
    if not fixed_file.is_file():
        raise FileNotFoundError('Complete fixed results are required before attaching human notes.')
    m94.atomic_json(m94.OUT/'fixed/human_review.json',HUMAN_NOTES)
    print('Saved:',m94.OUT/'fixed/human_review.json')
else:
    print('Write observations, then set SAVE_HUMAN_NOTES=True to store them.')


---

### Next milestone

Send `results/m9/m94_qualitative/fixed/m94_results.json`, `generations.csv` and `M94_READOUT.md`. Include `human_review.json` if completed. M9.5 will integrate M8 health, M9.3 independent NLL/PPL and M9.4 **both successful and unsuccessful** continuations, then decide whether any further pretraining is justified.
